In [84]:
!unzip 'a12c7399-5407-4af8-8b9d-681daea57e56'

In [94]:
import numpy as np
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

data = np.load('token_embeddings.npz')
tokens = data['tokens'].flatten() # We flatten the tokens matrix for safety
embeddings = data['embeddings']

# We reduce to 3D dimensionality (hint given on the platform "A cube is the shadow of a tesseract casted on 3 dimensions")
pca_3d = PCA(n_components=3, random_state=42)
puncte_3d = pca_3d.fit_transform(embeddings)

# We find the starting index (H)
all_H_indices = np.where(tokens == 'H')[0]

for start_idx in all_H_indices:
  vizited = [start_idx]
  remained_points = list(range(len(tokens)))
  remained_points.remove(start_idx)

  current_index = start_idx

  while remained_points:
    distances = cdist(puncte_3d[current_index].reshape(1, -1), puncte_3d[remained_points], 'euclidean').flatten()
    next_index = remained_points[np.argmin(distances)]
    vizited.append(next_index)
    remained_points.remove(next_index)
    current_index = next_index

  generated_text = "".join(tokens[vizited])


  if generated_text.startswith("HTB{"):
    print(generated_text)
    break

print(f"\n        Flag -> {generated_text[:23]}")

HTB{L0ST_1N_TH3_SP1R4L}7}SFDCE123____TYH{{DFPVAZX845CMNBVE}7!{FOIUTREQW4654#XZANMBLPEOVKGFEIDSU!}95HLPRI8WBR4-

        Flag -> HTB{L0ST_1N_TH3_SP1R4L}


In [89]:
import plotly.graph_objects as go
import pandas as pd

x_ordered = puncte_3d[vizited, 0]
y_ordered = puncte_3d[vizited, 1]
z_ordered = puncte_3d[vizited, 2]
ordered_tokens = tokens[vizited]

# We slice the arrays to keep only the first 23 elements (the clean flag)
x_flag = x_ordered[:23]
y_flag = y_ordered[:23]
z_flag = z_ordered[:23]
flag_tokens = ordered_tokens[:23]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x_flag,
    y=y_flag,
    z=z_flag,
    mode='lines+markers+text', # Draw lines, markers, and the text labels
    text=flag_tokens, # Uses the character letters as labels
    textposition="top center",
    textfont=dict(size=14, color="black", family="Courier New, monospace"),
    marker=dict(
        size=4,
        color=list(range(23)), # Color gradient mapping from 0 to 22
        colorscale='Viridis', # violet starts at 'H', and yellow starts at '}'
        opacity=0.9
    ),
    line=dict(
        color='royalblue', # Clean line linking the flag characters
        width=2
    ),
    name='Clean Flag Path'
))

fig.update_layout(
    title="The Vizualization of the Flag Itself - The Hidden 3D Shadow :)",
    scene=dict(
        xaxis_title="First Component (X)",
        yaxis_title="Second Component (Y)",
        zaxis_title="Third Component (Z)"
    ),
    width=900,
    height=700
)

fig.show()
